In [1]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
import pandas as pd
def create_splits(df, val_size=0.1, test_size=0.1):
    image_ids = df['image_name'].unique()

    train_ids, temp_ids = train_test_split(
        image_ids, test_size=(val_size + test_size), random_state=42
    )

    val_ids, test_ids = train_test_split(
        temp_ids, test_size=test_size / (val_size + test_size), random_state=42
    )

    return train_ids, val_ids, test_ids

In [2]:
# import os
# import pandas as pd
# import torch
# import torchvision
# from PIL import Image
# from torch.utils.data import Dataset


# class PollenDataset(Dataset):
#     def __init__(self, image_dir, csv_file, class_map, image_ids=None, transforms=None):
#         self.image_dir = image_dir
#         self.df = pd.read_csv(csv_file)
#         self.transforms = transforms
#         self.class_map = class_map

#         if image_ids is not None:
#             self.df = self.df[self.df['image_name'].isin(image_ids)]

#         self.image_ids = self.df['image_name'].unique()

#     def __len__(self):
#         return len(self.image_ids)

#     def __getitem__(self, idx):
#         image_id = self.image_ids[idx]
#         records = self.df[self.df['image_name'] == image_id]

#         img_path = os.path.join(self.image_dir, image_id)
#         image = Image.open(img_path).convert("RGB")

#         boxes = []
#         labels = []

#         for _, row in records.iterrows():
#             boxes.append([row['xmin'], row['ymin'], row['xmax'], row['ymax']])
#             labels.append(self.class_map[row['class_name']])

#         boxes = torch.tensor(boxes, dtype=torch.float32)
#         labels = torch.tensor(labels, dtype=torch.int64)

#         target = {"boxes": boxes, "labels": labels}

#         if self.transforms:
#             image = self.transforms(image)

#         return image, target

import os
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


class PollenDataset(Dataset):
    def __init__(self, image_dir, csv_file, class_map, image_ids=None, transforms=None):
        self.image_dir = image_dir
        self.df = pd.read_csv(csv_file)
        self.transforms = transforms
        self.class_map = class_map

        if image_ids is not None:
            self.df = self.df[self.df['image_name'].isin(image_ids)]
        self.df = self.df[self.df['image_name'].notna()]
        self.df = self.df[self.df['image_name'] != '#NAME?']
        self.df = self.df[self.df['image_name'] != '']

        self.image_ids = self.df['image_name']
        # Preload everything
        self.images = []
        self.targets = []

        for idx, image_id in enumerate(self.image_ids):
            records = self.df[self.df['image_name'] == image_id]
            if idx %500 == 0:
                print(idx, self.image_dir, image_id)
            img_path = os.path.join(self.image_dir, image_id)
            image = Image.open(img_path).convert("RGB")

            boxes = []
            labels = []

            for _, row in records.iterrows():
                boxes.append([row['xmin'], row['ymin'], row['xmax'], row['ymax']])
                labels.append(self.class_map[row['class_name']])

            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

            target = {"boxes": boxes, "labels": labels}

            # Apply transforms here if you want them fixed
            if self.transforms:
                image = self.transforms(image)

            self.images.append(image)
            self.targets.append(target)
            # self.data.append((image, target))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.targets[idx]
    
# IMAGE_DIR = r"pollen-20L/images"
# CSV_FILE = r"pollen-20L/bboxes.csv"
# CLASS_MAP_PATH = r"pollen-20L\class_map.csv"


# df = pd.read_csv(CLASS_MAP_PATH, index_col="class_name")
# class_map = df.to_dict()['index']

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ---- TRANSFORM ----
# transform = T.Compose([
#     T.Resize((300, 300)),
#     T.ToTensor()
# ])

# dataset = PollenDataset(
#     image_dir=IMAGE_DIR,
#     csv_file=CSV_FILE,
#     class_map=class_map,
#     transforms=transform
# )

In [ ]:
import torch
import torchvision
import torch
import torchvision
from torchvision.models.detection.ssd import SSDClassificationHead

import torch
import torchvision

class SSDVGGModel:
    def __init__(self, num_classes, device):
        self.device = device

        # IMPORTANT: +1 for background
        self.num_classes = num_classes + 1

       
        self.model = torchvision.models.detection.ssd300_vgg16(
                    weights=None,                    
                    weights_backbone="DEFAULT",     
                    num_classes=self.num_classes
                )

        self.model.to(self.device)

    def train_mode(self):
        self.model.train()

    def eval_mode(self):
        self.model.eval()

    def forward(self, images, targets=None):
        return self.model(images, targets)

    def parameters(self):
        return self.model.parameters()
# class SSDVGGModel:
#     def __init__(self, num_classes, device):
#         self.device = device
#         self.num_classes = num_classes

#         # Load pretrained SSD300 with VGG backbone
#         self.model = torchvision.models.detection.ssd300_vgg16(
#             weights="DEFAULT"
#         )

#         # Replace classification head
#         self._replace_head()

#         self.model.to(self.device)

#     def _replace_head(self):
#         # Torchvision expects num_classes including background
#         num_classes = self.num_classes + 1

#         in_channels = self.model.head.classification_head.num_classes
#         self.model.head.classification_head.num_classes = num_classes

#     def train_mode(self):
#         self.model.train()

#     def eval_mode(self):
#         self.model.eval()

#     def forward(self, images, targets=None):
#         return self.model(images, targets)

#     def parameters(self):
#         return self.model.parameters()

In [4]:
class Trainer:
    def __init__(self, model, dataloader, device, lr=1e-4):
        self.model = model
        self.dataloader = dataloader
        self.device = device
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

    def train_one_epoch(self):
        self.model.train_mode()
        total_loss = 0

        for idx, (images, targets) in enumerate(self.dataloader):
            images = [img.to(self.device) for img in images]
            targets = [{k: v.to(self.device) for k, v in t.items()} for t in targets]

            loss_dict = self.model.forward(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            self.optimizer.zero_grad()
            losses.backward()
            self.optimizer.step()

            total_loss += losses.item()
            print(f"Batch index : {idx} Loss = {losses.item()}")

        return total_loss

    def train(self, epochs):
        for epoch in range(epochs):
            loss = self.train_one_epoch()
            print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

In [5]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [6]:
import pandas as pd
df = pd.read_csv("pollen-20L\class_map.csv", index_col="class_name")
{key:value+1 for key, value in df.to_dict()['index'].items()}

{'buckwheat': 1,
 'clover': 2,
 'angelica': 3,
 'angelica_garden': 4,
 'willow': 5,
 'hill_mustard': 6,
 'linden': 7,
 'meadow_pink': 8,
 'alder': 9,
 'birch': 10,
 'fireweed': 11,
 'nettle': 12,
 'pigweed': 13,
 'plantain': 14,
 'sorrel': 15,
 'grass': 16,
 'pine': 17,
 'maple': 18,
 'hazel': 19,
 'mugwort': 20}

In [ ]:

IMAGE_DIR = r"pollen-20L/images"
CSV_FILE = r"pollen-20L/bboxes.csv"
CLASS_MAP_PATH = r"pollen-20L\class_map.csv"


df = pd.read_csv(CLASS_MAP_PATH, index_col="class_name")
class_map = {key:value+1 for key, value in df.to_dict()['index'].items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor()
])

dataset = PollenDataset(
    image_dir=IMAGE_DIR,
    csv_file=CSV_FILE,
    class_map=class_map,
    transforms=transform
)
dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

model = SSDVGGModel(
    num_classes=len(class_map),
    device=device
)

trainer = Trainer(
    model=model,
    dataloader=dataloader,
    device=device
)

trainer.train(epochs=10)

0 pollen-20L/images 443_png_jpg.rf.76a9b2e007453a66c8ffd722dbc009fd.jpg
500 pollen-20L/images ---------------------127_jpg.rf.8f0d73bd05171eb5b04af10504e36892.jpg
1000 pollen-20L/images 203_jpg.rf.508dc6206f090114873a47caf111fd94.jpg
1500 pollen-20L/images 183_jpg.rf.6355e7ae78473954a5bdc5cf77e6200e.jpg
2000 pollen-20L/images 25_jpg.rf.2419d056b2c99d975c1e4d0a1105ce66.jpg
2500 pollen-20L/images 940_png_jpg.rf.3953d7e1792e92c1d8ad82efb2b14215.jpg
3000 pollen-20L/images 193_jpg.rf.09351321c6095a9a14e5520ed41c8183.jpg
3500 pollen-20L/images 784_png_jpg.rf.a71dfba34501ba72d75ef34ca391e861.jpg
4000 pollen-20L/images --------------------136_jpg.rf.bcf9632993ed91f9c31384959c4d2c1c.jpg
4500 pollen-20L/images 86_jpg.rf.cea34be43ba87566fcf7e60aba3384b6.jpg
5000 pollen-20L/images 58_jpg.rf.e354fc5bdc8d68bba806497692fa702b.jpg
5500 pollen-20L/images 783_png_jpg.rf.feff56cc5ef59b2802507f4cab27a3f8.jpg
6000 pollen-20L/images 49_jpg.rf.96c8680d1e214dbc45e521b5af7cc4d8.jpg
6500 pollen-20L/images 945_p